# Explore R Simulation Techniques — Solution
## Monte-Carlo methods with base R (applied to race-place analysis)

Complete working solutions, alternates, extra practice answers, and a fully parameterised simulation section.

![Flowchart](r_simulation_techniques_flowchart.png)

![Example simulation plots](r_simulation_places_plots.png)


## 1. Reproducibility

In [ ]:
set.seed(42)
sample(1:10, 5)
# Expected (with seed 42): 10  9  3  6  4   (or similar depending on R version RNG)


## 2. Generative models

In [ ]:
set.seed(1)
cat("sample (discrete places):\n")
print(sample(1:30, size = 5, replace = TRUE))

cat("\nrunif:\n")
print(round(runif(5, 0, 1), 3))

cat("\nrnorm (approx place-like):\n")
print(round(rnorm(5, mean = 15, sd = 5), 1))


## 3. Single-trial function

In [ ]:
one_race <- function(n_runners = 30, friends = c("Megan", "Janet", "Tina")) {
  # Independent uniform places (simple model used throughout)
  places <- sample(1:n_runners, length(friends), replace = TRUE)
  names(places) <- friends
  places
}

set.seed(42)
one_race()
one_race(n_runners = 50)


## 4. Monte-Carlo repetition

In [ ]:
set.seed(42)
n_sims    <- 1000
n_runners <- 30
friends   <- c("Megan", "Janet", "Tina")

# Cleanest for a named vector return: replicate + t()
place_mat <- t(replicate(n_sims, one_race(n_runners, friends)))
colnames(place_mat) <- friends

cat("Dimensions:", dim(place_mat), "\n")
print(head(place_mat, 4))


## 5. Summaries

In [ ]:
cat("Expected place (column means):\n")
print(round(colMeans(place_mat), 2))

cat("\nQuantiles:\n")
print(apply(place_mat, 2, quantile, probs = c(0.25, 0.5, 0.75)))

cat("\nP(place <= 10):\n")
print(round(colMeans(place_mat <= 10), 3))


## 6. Visualization (base R)

In [ ]:
# In a real R session the plots appear interactively.
# Example code:
# par(mfrow = c(1, 3))
# for (i in seq_along(friends)) {
#   hist(place_mat[, i], breaks = 15, main = friends[i],
#        xlab = "Place", col = "steelblue", border = "white")
#   abline(v = mean(place_mat[, i]), col = "red", lwd = 2, lty = 2)
# }
# boxplot(place_mat, names = friends, col = c("#2E86AB", "#A23B72", "#F18F01"),
#         main = "Place distributions", ylab = "Place")
# See also the PNG generated for the report: r_simulation_places_plots.png


## 7. Sensitivity to field size

In [ ]:
set.seed(99)
n_runners_grid <- c(15, 30, 60, 100)
top10_by_n <- sapply(n_runners_grid, function(nr) {
  mat <- t(replicate(800, one_race(nr, friends)))
  colMeans(mat <= 10)
})
colnames(top10_by_n) <- paste0("n=", n_runners_grid)
print(round(top10_by_n, 3))

# Interpretation: as the field grows, P(top-10) falls roughly like 10/n_runners


## Alternate Techniques — Bootstrap

In [ ]:
# Non-parametric bootstrap of mean place for Megan
megan_places <- place_mat[, "Megan"]
set.seed(7)
boot_means <- replicate(2000, mean(sample(megan_places, replace = TRUE)))

cat("Original mean:", mean(megan_places), "\n")
cat("Bootstrap SE  :", sd(boot_means), "\n")
cat("Approx 95% CI :", round(quantile(boot_means, c(0.025, 0.975)), 2), "\n")


## More Practice — Solutions

In [ ]:
# 1. Truncated-normal places (still integer)
one_race_norm <- function(n_runners = 30, friends = c("Megan", "Janet", "Tina")) {
  raw <- rnorm(length(friends), mean = n_runners/2, sd = n_runners/4)
  places <- pmax(1, pmin(n_runners, round(raw)))
  names(places) <- friends
  places
}
set.seed(42)
mat_norm <- t(replicate(1000, one_race_norm(30, friends)))
cat("Means under truncated-normal model:\n")
print(round(colMeans(mat_norm), 2))

# 2. Probability all three in top half
top_half <- n_runners / 2
p_all_top_half <- mean(apply(place_mat <= top_half, 1, all))
cat("\nP(all three in top half):", round(p_all_top_half, 3), "\n")

# 3. Best place among the three friends
best_place <- apply(place_mat, 1, min)
cat("Mean of best place:", round(mean(best_place), 2), "\n")
cat("P(best place <= 5):", round(mean(best_place <= 5), 3), "\n")


## Interactive Simulation Section (ready to tweak)
Change any of the parameters and re-execute the cell.


In [ ]:
# === PARAMETERS ===
set.seed(123)
n_runners   <- 40
n_sims      <- 1500
friends     <- c("Megan", "Janet", "Tina")
success_at  <- 10

# === ENGINE ===
one_race <- function(n_runners, friends) {
  places <- sample(1:n_runners, length(friends), replace = TRUE)
  names(places) <- friends
  places
}

place_mat <- t(replicate(n_sims, one_race(n_runners, friends)))
colnames(place_mat) <- friends

cat("n_runners =", n_runners, "| n_sims =", n_sims, "\n")
cat("Expected places:\n"); print(round(colMeans(place_mat), 2))
cat("P(place <=", success_at, "):\n"); print(round(colMeans(place_mat <= success_at), 3))
cat("P(all three <=", success_at, "):", round(mean(apply(place_mat <= success_at, 1, all)), 3), "\n")


## Key Results Snapshot (default settings n=30, 1000 sims)
- Expected place for each friend ≈ 15.5 (as expected under uniform)
- P(place ≤ 10) ≈ 0.33 for each friend independently
- As field size grows, top-k probability declines approximately as k / n_runners
- Bootstrap gives a simple non-parametric confidence interval for any summary statistic
